# gorz — DuckDB community extension smoke tests

[`gorz`](https://duckdb.org/community_extensions/extensions/gorz) reads and writes
[GORpipe](https://github.com/gorpipe/gor) genomic files (`.gorz`, `.gord`) as native
DuckDB tables. This notebook installs it **straight from the community repository**
(nothing local, signed, no `-unsigned`) and exercises the core features.

**Requires DuckDB ≥ 1.5.4** — the version the community pipeline currently targets.


In [1]:
import duckdb
print("duckdb", duckdb.__version__)

con = duckdb.connect()
con.execute("INSTALL gorz FROM community; LOAD gorz;")

# Confirm it came from the community REPOSITORY (signed), not a local build.
con.sql("""
    SELECT extension_name, installed, install_mode
    FROM duckdb_extensions() WHERE extension_name = 'gorz'
""").df()


duckdb 1.5.4


,extension_name,installed,install_mode
0,gorz,True,REPOSITORY


## 1. `.gorz` round-trip

Write a result set out as a block-compressed `.gorz` (rows must be in GOR order —
chromosomes lexicographic, positions non-decreasing) and read it back.


In [2]:
con.execute("""
    COPY (SELECT * FROM (VALUES
            ('chr1', 1000, 'A', 'G'),
            ('chr1', 2000, 'C', 'T'),
            ('chr2',  500, 'G', 'C'))
          t(chrom, pos, ref, alt) ORDER BY chrom, pos)
      TO '/tmp/variants.gorz' (FORMAT gorz);
""")

con.sql("SELECT * FROM read_gor('/tmp/variants.gorz')").df()


,chrom,pos,ref,alt
0,chr1,1000,A,G
1,chr1,2000,C,T
2,chr2,500,G,C


## 2. Range pushdown — GOR's `-p` semantics

`range := 'chrN:start-end'` is a hard positional filter (also `'chrN'`,
`'chrN:start-'`, `'chrN:pos'`); it seeks into the right block instead of scanning.


In [3]:
con.sql("SELECT * FROM read_gor('/tmp/variants.gorz', range := 'chr1:1500-2500')").df()


,chrom,pos,ref,alt
0,chr1,2000,C,T


## 3. `.gord` dictionary + the exposed `Source` column

A `.gord` is a dictionary of per-tag `.gorz` files. `read_gord` merges them in
genomic order and appends a trailing **`Source`** column naming each row's origin
tag (column 2 / the alias) — matching gorpipe. Here we build a two-file dictionary
entirely with `COPY` (the pipe-bearing file column is just a string value).


In [4]:
con.execute("""
    COPY (SELECT * FROM (VALUES ('chr1',10,'A'),('chr1',30,'A')) t(chrom,pos,val) ORDER BY chrom,pos)
      TO '/tmp/s1.gorz' (FORMAT gorz);
    COPY (SELECT * FROM (VALUES ('chr1',20,'B'),('chr1',40,'B')) t(chrom,pos,val) ORDER BY chrom,pos)
      TO '/tmp/s2.gorz' (FORMAT gorz);
    COPY (SELECT * FROM (VALUES
            ('s1.gorz','s1','chr1',1,'chr1',1000000),
            ('s2.gorz','s2','chr1',1,'chr1',1000000)) t(f,tag,cs,ps,ce,pe))
      TO '/tmp/d.gord' (FORMAT csv, DELIMITER E'\t', HEADER false);
""")

con.sql("SELECT chrom, pos, val, Source FROM read_gord('/tmp/d.gord') ORDER BY pos").df()


,chrom,pos,val,Source
0,chr1,10,A,s1
1,chr1,20,B,s2
2,chr1,30,A,s1
3,chr1,40,B,s2


## 4. Partition filtering — `-f` / `-ff`

`f := [...]` / `ff := [...]` prune the dictionary to the requested tags (GOR's
`-f`/`-ff`). With no column-7 tag list, the filter keys off column 2 (the alias).


In [5]:
con.sql("SELECT * FROM read_gord('/tmp/d.gord', f := ['s2']) ORDER BY pos").df()


,chrom,pos,val,Source
0,chr1,20,B,s2
1,chr1,40,B,s2


## 5. Tag files (`ff`) — built from a query with `COPY`

`f` takes an inline `LIST(VARCHAR)`; `ff` takes a path to a **tag file** — a
single-column `.tsv` (first tab-delimited column, `#` lines skipped). A tidy way
to build one is to `COPY` a `SELECT`, so a cohort query *becomes* the filter:

*`ff`-as-file ships in **v0.1.1** (PR #2225). The published v0.1.0 treats `ff`
like `f`, so the cell below previews it against the in-tree local build; once
v0.1.1 is published, `INSTALL gorz FROM community` provides it and you'd use the
`con` connection directly.*


In [6]:
# Build the tag file with pure SQL: COPY a SELECT (a cohort) to a 1-column .tsv
con.execute("CREATE OR REPLACE TABLE cohort(tag) AS SELECT * FROM (VALUES ('s1'), ('s2'));")
con.execute("COPY (SELECT tag FROM cohort WHERE tag <> 's1') TO '/tmp/tags.tsv' (FORMAT csv, HEADER false)")
print("tags.tsv:\n" + open('/tmp/tags.tsv').read())

# ff := <tag file> selects rows by tag, reading the cohort from the file. The
# f := ['s2'] form above takes the same tags inline.
con.sql(
    "SELECT chrom, pos, val, Source FROM read_gord('/tmp/d.gord', ff := '/tmp/tags.tsv') ORDER BY pos"
).df()

tags.tsv:
s2



,chrom,pos,val,Source
0,chr1,20,B,s2
1,chr1,40,B,s2


---
### Notes
- Installed with `INSTALL gorz FROM community; LOAD gorz;` — no local build, no `-unsigned`.
- Also available: `read_gorz(path)` / `read_gord(path)` force the kind; a bare
  `'file.gorz'` literal works as a replacement scan; `s3://…` paths resolve when
  `httpfs` is loaded.
- This is the published **v0.1.0**. Docs: <https://duckdb.org/community_extensions/extensions/gorz>
